In [ ]:
# install the source_table branch for now
!git clone -b source_table --single-branch https://github.com/james-trayford/strauss.git
!pip install "strauss @ git+https://github.com/james-trayford/strauss.git@source_table" -q
%cd strauss/examples/

# Source Table examples

We now make timing tables from the `Sources` to allow for syncing (e.g. visuals) or just general inspection. Set up some data first:

In [ ]:
import strauss
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
x = np.linspace(0, 90, 400)
y =  np.sin(x/2)
z = 0.1*x**2 - x - 5 

### Basic Tables

Try **`default`** sonification with nothing added:   

In [ ]:
strauss.sonify(x, y)
mapped = strauss.get_table()
fixed = strauss.get_fixed_table()
strauss.display()
print("\n\nMapped sources table:")
strauss.list_tables()
strauss.close()

This produces a single _"object"_ source with a single pitch-mapping. For `Objects`, each row is a single data point with columns denoting property value at each time.

In [ ]:

# show mapped sources table (just one in this case)
mapped 

In [ ]:
# show fixed sources table
fixed

If I have multiple sonifications, I can get the table for each one by passing the sonification name to `get_table()`. For example:

In [ ]:
strauss.sonify(x, y)
strauss.sonify(x, -y, style='mallets')
afig = strauss._get_current_figure()
strauss.display()
strauss.close()

In [ ]:
afig.list_tables()

In [ ]:
afig.get_table('sonification_2')

In [ ]:
afig.get_table('sonification_2')

### `Events` Sources and naming

First, lets generate a load of names for the sources. Using IAU star names to generate a list of nouns of length `len(x)`...

In [ ]:
import pandas as pd
import random

In [ ]:
!curl https://raw.githubusercontent.com/cyschneck/iau-star-names/main/iau_proper_stars.csv > iau_proper_stars.csv
names = sorted(random.sample(list(pd.read_csv("iau_proper_stars.csv")["Proper Names"]), len(x)))

First without explicit naming let's make a `mallets` sonification. 

In [ ]:

soni =  strauss.sonify(x, y, 
                       style='mallets')
display(strauss.get_table())
strauss.display()
strauss.close()

Note that rendering the sonification itself is not necessary to produce the tables.

Now with the names, and we'll reverse all arrays  to validate the tables should always return in time order....

In [ ]:
soni =  strauss.sonify(x[::-1], y[::-1], 
                       style='mallets',
                       source_names=names[::-1])
display(strauss.get_table())
strauss.close()

For `Events`, each row is a single event with columns denoting properties at their respective occurence time. Printing our first few input names for comparison:

In [ ]:
print(names[:10])

...notice that this isn't a complete list of provided names. This is because the `max_num_per_sec` assigned by the style merges some sources together. 

Now with named sources this can be problematic - if esch source is a particular star, say, we don't don't want to represent them as an 'average' star at an 'average' position in the sky (e.g. for _Betelgeuse_ and _Sirius_ an average of their colour, position, etc).

This is why the **new default merge approach** is to take the average ***time*** an event sounds, but then take all the other properties from the entry closest in time to the mid point of the merged sources. 

We can use the old approach using a _kwarg_ or style parameter...

In [ ]:
soni =  strauss.sonify(x, y, 
                       style='mallets', 
                       source_names=names,
                       merge_mode='average', # <== this is the new parameter
                       num_notes_per_sec=1)
afig = strauss._get_current_figure()
strauss.close()

now the names reflect that they are the average of multiple sources. The name is listed as `{earliest}->{latest}` to indicate the names of the earliest and latest source occuring in the cluster

In [ ]:
afig.get_table('sonification_1')

### Complex events with spatial angles

Finally, sources with spatial angles use updated angle behaviour. The style file and kwargs now have `angle_unit` parameter I can set to work in a particular unit convention (where for a full circle _'cycles'_ expects 0->1, _'degrees'_ expects 0->360 and _'radians'_ expects 0->2π)

This unit is what is expected for spatial angle fields, allowing > the limits to be used (e.g. 720 degrees). If set but the input and table units willuse that convention, if unset we expect cycles (following ~old behaviour) but tabulate in degrees.

Lets wrap this around the listener twice (azimuth: 0 -> 720 degrees) in the listener plane (polar: 90 degrees)

In [ ]:
windy = strauss.sonify(x, # time
                       y, # pitch
                       720*x/x.max(), # azimuth angle
                       x*0 + 90, # polar angle
                       angle_unit='degrees',
                       style='stars_appearing', 
                       source_names=names,
                       merge_mode='central')
display(strauss.get_table("sonification_1"))
strauss.display()
strauss.close()